# 04 · Mappings and knowledge graph – Nordvia Retail (SAP FI)

**Goal.** Run the mappings over the gold tables to build the knowledge graph, check it against the ontology and against gold, and save it.

**Pieces**
- `mappings/nordvia.yarrrml.yml` – the mappings, in YARRRML: one block per gold table, each line a column and the property it becomes. They hold no logic; everything was decided in gold.
- `mappings/config.ini` – tells Morph-KGC where the mappings are.
- `ontology/nordvia-finance.ttl` – the ontology the graph has to conform to.

**How to rebuild the graph from the CSVs.** Run the notebooks in order: `02_silver` → `03_gold` → this one. Outside a notebook, the last step is `python -m morph_kgc mappings/config.ini`.

## 0 · Environment

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
%pip install -q morph-kgc rdflib

import os, sys, collections
from importlib.metadata import version
import pandas as pd
import morph_kgc
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS

pd.set_option("display.width", 220)
print("Python", sys.version.split()[0], "|", ", ".join(f"{p} {version(p)}" for p in ["morph-kgc", "rdflib", "pandas"]))

NVF = Namespace("https://nordvia.com/ontology/finance#")
ID = "https://nordvia.com/id/"
GOLD = lambda name: pd.read_csv(f"gold/{name}.csv", dtype=str, keep_default_na=False)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: c:\Users\Javier Granda\.pyenv\pyenv-win\versions\3.12.7\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Python 3.12.7 | morph-kgc 2.10.0, rdflib 7.2.1, pandas 2.3.3


## 1 · Build the graph
Morph-KGC reads the mappings, goes through the gold tables and returns the triples as an RDFLib graph.

In [3]:
kg = morph_kgc.materialize("mappings/config.ini")
print(f"{len(kg):,} triples")

119,299 triples


## 2 · Checks
### 2.1 The graph only uses terms of the ontology
Every property, every class and every individual of the ontology that appears in the graph must be declared in `nordvia-finance.ttl`. This catches a misspelt term in the mappings, which would otherwise produce triples nobody can query.

In [4]:
onto = Graph().parse("ontology/nordvia-finance.ttl", format="turtle")
declared = set(onto.subjects(RDF.type, None))

properties = set(kg.predicates()) - {RDF.type, RDFS.label}
classes = set(kg.objects(None, RDF.type))
individuals = {o for o in kg.objects() if isinstance(o, URIRef) and not str(o).startswith(ID)} - classes

unknown = sorted(t for t in properties | classes | individuals if t not in declared)
assert not unknown, unknown
print(len(properties), "properties,", len(classes), "classes and", len(individuals), "individuals used; all declared in the ontology")

70 properties, 36 classes and 25 individuals used; all declared in the ontology


### 2.2 No broken links
Every instance the graph points to must exist in the graph, and no identifier may be empty or malformed.

In [5]:
subjects = set(kg.subjects())
dangling = {o for o in kg.objects() if isinstance(o, URIRef) and str(o).startswith(ID) and o not in subjects}
malformed = [str(t) for t in subjects if str(t).endswith(("/", "nan", "None")) or " " in str(t)]
assert not dangling and not malformed, (list(dangling)[:5], malformed[:5])
print(len(subjects), "instances; no dangling links, no malformed identifiers")

9049 instances; no dangling links, no malformed identifiers


### 2.3 The graph has what gold has
Instances per class against rows in gold, links against filled cells, and the total local amount.

In [6]:
per_class = collections.Counter(str(c).split("#")[-1] for c in kg.objects(None, RDF.type))
li, docs = GOLD("line_item"), GOLD("accounting_document")

def links(prop, only=None):
    return sum(1 for s in kg.subjects(NVF[prop], None) if only is None or only in str(s))

rows = {
    "Company": (per_class["Company"], len(GOLD("company"))),
    "GLAccount": (per_class["GLAccount"], len(GOLD("gl_account"))),
    "CompanyAccount": (per_class["CompanyAccount"], len(GOLD("company_account"))),
    "CostCenter": (per_class["CostCenter"], len(GOLD("cost_center"))),
    "Customer": (per_class["Customer"], len(GOLD("customer"))),
    "Payee": (per_class["Payee"], len(GOLD("payee"))),
    "Address": (per_class["Address"], len(GOLD("address"))),
    "AccountingDocument": (per_class["AccountingDocument"], len(docs)),
    "LineItem": (per_class["LineItem"], len(li)),
    "MonthlyExchangeRate": (per_class["MonthlyExchangeRate"], len(GOLD("exchange_rate"))),
    "line -> customer": (links("customer"), int((li["customer_number"] != "").sum())),
    "line -> payee": (links("payee"), int((li["payee_number"] != "").sum())),
    "line -> cost center": (links("costCenter"), int((li["cost_center_number"] != "").sum())),
    "line -> clearing document": (links("clearedBy"), int((li["cleared_by_document_id"] != "").sum())),
    "line with due date": (links("dueDate"), int((li["due_date"] != "").sum())),
    "line with amount in euros": (links("amountInGroupCurrency"), len(li)),
    "credit memo -> invoice": (links("creditFor"), int((docs["credit_for_document_id"] != "").sum())),
    "reversal -> original": (links("reverses"), int((docs["reverses_document_id"] != "").sum())),
    "total local amount": (round(sum(float(v) for v in kg.objects(None, NVF.amountInLocalCurrency)), 2),
                           round(pd.to_numeric(li["amount_local_currency"]).sum(), 2)),
}
check = pd.DataFrame(rows, index=["graph", "gold"]).T
check["ok"] = check["graph"] == check["gold"]
assert check["ok"].all(), check[~check["ok"]]
check

,graph,gold,ok
Company,3.00,3.00,True
GLAccount,27.00,27.00,True
CompanyAccount,80.00,80.00,True
CostCenter,25.00,25.00,True
Customer,157.00,157.00,True
Payee,71.00,71.00,True
Address,230.00,230.00,True
AccountingDocument,2353.00,2353.00,True
LineItem,6062.00,6062.00,True
MonthlyExchangeRate,24.00,24.00,True


### 2.4 What one instance looks like
The customer invoice whose reference collides with another invoice (DQ-12), as it is stored in the graph.

In [7]:
def describe(iri):
    short = lambda t: t.n3(kg.namespace_manager) if isinstance(t, Literal) else str(t).replace(str(NVF), "nvf:").replace(ID, "id:")
    return pd.DataFrame(sorted((short(p).split("/")[-1].split("#")[-1], short(o)) for p, o in kg.predicate_objects(URIRef(iri))),
                        columns=["property", "value"])

describe(ID + "line-item/2000-1800000014-2025-001")

,property,value
0,description,"""Sales invoice"""
1,nvf:accountType,nvf:CustomerAccountType
2,nvf:amountInDocumentCurrency,"""8289.71""^^xsd:decimal"
3,nvf:amountInGroupCurrency,"""8034.39""^^xsd:decimal"
4,nvf:amountInLocalCurrency,"""8289.71""^^xsd:decimal"
5,nvf:baselineDate,"""2025-01-23T00:00:00""^^xsd:dateTime"
6,nvf:clearedBy,id:document/2000-1400000064-2025
7,nvf:clearingDate,"""2025-05-21T00:00:00""^^xsd:dateTime"
8,nvf:customer,id:customer/0000100108
9,nvf:dueDate,"""2025-03-24T00:00:00""^^xsd:dateTime"


## 3 · Save
The graph is saved as Turtle in `kg/nordvia-kg.ttl`. It holds the data only; the ontology stays in its own file and the two are loaded together to query (section 4).

In [8]:
PREFIXES = {"nvf": str(NVF), "dcterms": "http://purl.org/dc/terms/", "skos": "http://www.w3.org/2004/02/skos/core#",
            "foaf": "http://xmlns.com/foaf/0.1/", "org": "http://www.w3.org/ns/org#", "rov": "http://www.w3.org/ns/regorg#",
            "locn": "http://www.w3.org/ns/locn#", "sdo": "https://schema.org/",
            "fibo-4217": "https://spec.edmcouncil.org/fibo/ontology/FND/Accounting/ISO4217-CurrencyCodes/"}
for kind in ["company", "account", "company-account", "cost-center", "customer", "payee", "address", "document", "line-item",
             "document-type", "posting-key", "exchange-rate", "chart-of-accounts", "controlling-area"]:
    PREFIXES[kind] = f"{ID}{kind}/"
for prefix, namespace in PREFIXES.items():
    kg.bind(prefix, namespace, override=True)

os.makedirs("kg", exist_ok=True)
kg.serialize("kg/nordvia-kg.ttl", format="turtle")
print(f"{len(kg):,} triples saved to kg/nordvia-kg.ttl ({os.path.getsize('kg/nordvia-kg.ttl') / 1e6:.1f} MB)")

119,299 triples saved to kg/nordvia-kg.ttl (4.7 MB)


## 4 · A first query
Data and ontology are loaded into one graph. With the ontology present, a query can ask for a general class and follow `rdfs:subClassOf` down to the specific ones, without a reasoner: here, every kind of accounting document.

In [9]:
full = Graph()
full += kg
full += onto

q = '''
PREFIX nvf: <https://nordvia.com/ontology/finance#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?kind (COUNT(DISTINCT ?doc) AS ?documents)
WHERE {
  ?class rdfs:subClassOf+ nvf:AccountingDocument ; rdfs:label ?kind .
  ?doc a/rdfs:subClassOf* ?class .
}
GROUP BY ?kind ORDER BY DESC(?documents)
'''
pd.DataFrame([(str(r.kind), int(r.documents)) for r in full.query(q)], columns=["kind of document", "documents"])

,kind of document,documents
0,Invoice,1164
1,Customer Invoice,683
2,Customer Payment,590
3,Payable Invoice,481
4,Outgoing Payment,422
5,General Ledger Document,148
6,Customer Credit Memo,22
7,Reversal Document,7
8,Opening Balance Document,3


No document is typed as *Invoice* in the data, yet the query counts the customer and payable invoices under it; in the same way *General Ledger Document* includes the opening balance documents. A document can therefore be counted in more than one row. The five competency questions are answered on this same combined graph.